# US Major Cities Map

Interactive scatter map of 227+ major US cities with state labels, built with Plotly.

**Data source:** [kapkaev/major_us_city_dma_codes](https://gist.github.com/kapkaev/a67b6758606c0a5e11af)

In [4]:
import requests
import pandas as pd
import plotly.express as px
import geopandas as gpd


In [5]:
#US cities dataset 
url = "https://gist.githubusercontent.com/kapkaev/a67b6758606c0a5e11af/raw/major_us_city_dma_codes.py"
resp = requests.get(url)
exec(resp.text)

cities_df = pd.DataFrame(major_cities)
print(f"Loaded {len(cities_df)} cities")
cities_df.head()

Loaded 227 cities


,city,dma_code,latitude,longitude,region,slug
0,Ada,657,34.774531,-96.678345,OK,ada-ok
1,Akron,510,41.081445,-81.519005,OH,akron-oh
2,Albany,525,31.578507,-84.155741,GA,albany-ga
3,Alexandria,644,31.311294,-92.445137,LA,alexandria-la
4,Alpena,583,45.061679,-83.432753,MI,alpena-mi


In [15]:
#Center coordinates for each US state
gdf = gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_5m.zip")
gdf['centroid_lat'] = gdf.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).y
gdf['centroid_lon'] = gdf.geometry.to_crs(epsg=3857).centroid.to_crs(epsg=4326).x

# só os estados que têm cidades no teu dataset
states_in_data = cities_df['region'].unique()
gdf_filtered = gdf[gdf['STUSPS'].isin(states_in_data)]

us_states = dict(zip(gdf_filtered['NAME'], zip(gdf_filtered['centroid_lat'], gdf_filtered['centroid_lon'])))

In [16]:
# scatter map


fig = px.scatter_geo(
    cities_df,
    lat='latitude',
    lon='longitude',
    hover_name='city',
    hover_data={'region': True, 'dma_code': True, 'latitude': ':.2f', 'longitude': ':.2f'},
    title=f'US Cities ({len(cities_df)} cities)',
    scope='usa',
    projection='albers usa'
)

# Add state name labels
for state, (lat, lon) in us_states.items():
    fig.add_scattergeo(
        lat=[lat], lon=[lon],
        text=[state],
        mode='text',
        textfont=dict(size=12, color='black', family='Arial'),
        showlegend=False,
        hoverinfo='skip'
    )

# Style the map
fig.update_layout(
    geo=dict(
        scope='usa',
        projection_type='albers usa',
        showland=True,
        showlakes=True,
        landcolor='rgb(220, 208, 240)',
        lakecolor='rgb(204, 229, 255)',
        subunitcolor='white',
        subunitwidth=1.5
    ),
    height=600,
    margin={"r": 0, "t": 30, "l": 0, "b": 0}
)

fig.show()